In [1]:
// Añadimos Apache Spark Core y SQL para Scala 2.13
// El sufijo _2.13 es gestionado automáticamente por $ivy cuando usamos %%
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

println("✅ Dependencias de Spark cargadas correctamente")

✅ Dependencias de Spark cargadas correctamente


import $ivy.$
import $ivy.$

In [2]:
import org.apache.log4j.{Level, Logger}

// Silenciamos los logs de Spark para que el output sea legible
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

println("✅ Logs de Spark configurados (solo se mostrarán errores)")

✅ Logs de Spark configurados (solo se mostrarán errores)


import org.apache.log4j.{Level, Logger}

In [3]:
import org.apache.spark.sql.SparkSession

// Creamos la SparkSession en modo local
// "local[*]" significa: usa todos los núcleos de CPU disponibles
val spark = SparkSession.builder()
  .appName("IntroSpark")   // nombre visible en el Spark UI
  .master("local[*]")                    // modo local, todos los núcleos
  .config("spark.ui.showConsoleProgress", "false")  // sin barras de progreso
  .getOrCreate()

// Obtenemos el SparkContext desde la sesión
val sc = spark.sparkContext

println(s"✅ SparkSession creada correctamente")
println(s"   Versión de Spark: ${spark.version}")
println(s"   Nombre de la app: ${sc.appName}")
println(s"   Master:           ${sc.master}")
println(s"   Spark UI:         http://localhost:4040")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:21:21 INFO SparkContext: Running Spark version 4.1.1
26/04/22 11:21:21 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/22 11:21:21 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/04/22 11:21:21 INFO ResourceUtils: ==============================================================
26/04/22 11:21:21 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/22 11:21:21 INFO ResourceUtils: ==============================================================
26/04/22 11:21:21 INFO SparkContext: Submitted application: IntroSpark
26/04/22 11:21:22 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/22 11:21:22 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/22 11:21:22 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/22 11:21:22 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/22 11:21:22 INFO SecurityManager: Changing view acls to

2026-04-22T09:21:22.362056500Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@1fbeb097
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@5005b38a

In [4]:
// Creamos un RDD con números del 1 al 1.000.000
val numeros = sc.parallelize(1 to 1000000)
println(s"Particiones creadas: ${numeros.getNumPartitions}")

Particiones creadas: 8


numeros: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[0] at parallelize at cmd4.sc:2

In [5]:
// Esta acción dispara el primer Job
val total = numeros.count()
println(s"Total elementos: $total")

Total elementos: 1000000


total: Long = 1000000L

🔍 Ve a la Spark UI ahora. En la pestaña Jobs verás que ha aparecido un job nuevo llamado count. Haz clic en él para ver los Stages y las Tasks. Anota: ¿cuántos stages tiene? ¿Cuántas tasks?
R: s = 8
Stages=1


Paso 3 — Segunda acción: filter + count (dos transformaciones, un job)

In [6]:
// Transformación 1: filtrar pares (lazy, no ejecuta)
val pares = numeros.filter(_ % 2 == 0)

// Transformación 2: multiplicar por 3 (lazy, no ejecuta)
val paresPorTres = pares.map(_ * 3)

// ACCIÓN: ahora sí ejecuta todo el plan
val resultado = paresPorTres.count()
println(s"Cantidad de (pares × 3): $resultado")

Cantidad de (pares × 3): 500000


pares: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[1] at filter at cmd6.sc:2
paresPorTres: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[2] at map at cmd6.sc:5
resultado: Long = 500000L

Vuelve a la Spark UI. Ha aparecido un segundo job. Compara su estructura con el anterior: ¿tiene más stages? ¿Por qué?
R: stages=2
tasks=8

Paso 4 — Tercera acción: reducir a un único valor

In [7]:
// Suma de todos los números pares multiplicados por 3
val suma = paresPorTres.reduce(_ + _)
println(s"Suma total: $suma")

Suma total: -1617776800


suma: Int = -1617776800

🔍 En la Spark UI este job tendrá 2 stages. ¿Por qué? Porque reduce necesita un shuffle parcial: primero cada executor suma su parte (Stage 1) y luego el Driver combina los resultados parciales (Stage 2).

🔹 Ejercicio 3 — Visualizar el DAG
Paso 1 — Crear una cadena de transformaciones:

In [8]:
val ventas = sc.parallelize(List(
  ("Madrid", 1200.0),
  ("Barcelona", 800.0),
  ("Madrid", 950.0),
  ("Valencia", 600.0),
  ("Barcelona", 1100.0),
  ("Madrid", 700.0),
  ("Valencia", 850.0),
  ("Barcelona", 300.0)
))

// Transformaciones (ninguna ejecuta todavía)
val ventasMadrid    = ventas.filter(_._1 == "Madrid")
val importesMadrid  = ventasMadrid.map(_._2)
val totalMadrid     = importesMadrid.reduce(_ + _)

println(f"Total ventas Madrid: $totalMadrid%.2f €")

Total ventas Madrid: 2850,00 €


ventas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[3] at parallelize at cmd8.sc:1
ventasMadrid: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[4] at filter at cmd8.sc:13
importesMadrid: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[5] at map at cmd8.sc:14
totalMadrid: Double = 2850.0

Paso 2 — Ver el linaje del RDD:

In [9]:
// toDebugString muestra el DAG en texto
println(importesMadrid.toDebugString)

(8) MapPartitionsRDD[5] at map at cmd8.sc:14 []
 |  MapPartitionsRDD[4] at filter at cmd8.sc:13 []
 |  ParallelCollectionRDD[3] at parallelize at cmd8.sc:1 []


Paso 3 — Un ejemplo con shuffle (operación por clave):

In [10]:
// groupByKey fuerza un shuffle → límite entre stages
val ventasPorCiudad = ventas.groupByKey()

// toDebugString antes de la acción
println(ventasPorCiudad.toDebugString)

(8) ShuffledRDD[6] at groupByKey at cmd10.sc:2 []
 +-(8) ParallelCollectionRDD[3] at parallelize at cmd8.sc:1 []


ventasPorCiudad: org.apache.spark.rdd.RDD[(String, Iterable[Double])] = ShuffledRDD[6] at groupByKey at cmd10.sc:2

In [12]:
val ventasLista = ventas.collect()

val resumen = ventasLista
  .groupBy(_._1)
  .mapValues(_.map(_._2).sum)

// 2. Imprimimos el resultado
println("Resumen de ventas por ciudad (Procesamiento local):")
resumen.foreach { case (ciudad, total) =>
  println(f"  $ciudad%-15s → $total%.2f €")
}

1 deprecation (since 2.13.0); re-run enabling -deprecation for details, or try -help


Resumen de ventas por ciudad (Procesamiento local):
  Barcelona       → 2200,00 €
  Madrid          → 2850,00 €
  Valencia        → 1450,00 €


ventasLista: Array[(String, Double)] = Array(
  ("Madrid", 1200.0),
  ("Barcelona", 800.0),
  ("Madrid", 950.0),
  ("Valencia", 600.0),
  ("Barcelona", 1100.0),
  ("Madrid", 700.0),
  ("Valencia", 850.0),
  ("Barcelona", 300.0)
)
resumen: collection.MapView[String, Double] = MapView(
  ("Barcelona", 2200.0),
  ("Madrid", 2850.0),
  ("Valencia", 1450.0)
)

🔹 Ejercicio 4 — Identificar stages y tasks
Ejecuta el siguiente código y luego responde las preguntas usando la Spark UI:

In [15]:
val palabras = sc.parallelize(List(
  "spark", "scala", "big", "data", "spark", "es", "rapido",
  "scala", "es", "elegante", "spark", "escala", "bien",
  "big", "data", "necesita", "spark"
))

// 1. Traemos las palabras a la memoria local con collect()
val listaPalabras = palabras.collect()

// 2. Usamos Scala estándar para contar y ordenar
val frecuencias = listaPalabras
  .groupBy(identity)
  .mapValues(_.size)
  .toList
  .sortBy(-_._2) // El signo menos ordena de mayor a menor

// 3. Imprimimos el resultado
println("Frecuencia de palabras (Procesamiento local):")
frecuencias.foreach { case (palabra, count) =>
  println(f"  $palabra%-15s: $count veces")
}


1 deprecation (since 2.13.0); re-run enabling -deprecation for details, or try -help


Frecuencia de palabras (Procesamiento local):
  spark          : 4 veces
  big            : 2 veces
  data           : 2 veces
  scala          : 2 veces
  es             : 2 veces
  necesita       : 1 veces
  escala         : 1 veces
  rapido         : 1 veces
  bien           : 1 veces
  elegante       : 1 veces


palabras: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[21] at parallelize at cmd15.sc:1
listaPalabras: Array[String] = Array(
  "spark",
  "scala",
  "big",
  "data",
  "spark",
  "es",
  "rapido",
  "scala",
  "es",
  "elegante",
  "spark",
  "escala",
  "bien",
  "big",
  "data",
  "necesita",
  "spark"
)
frecuencias: List[(String, Int)] = List(
  ("spark", 4),
  ("big", 2),
  ("data", 2),
  ("scala", 2),
  ("es", 2),
  ("necesita", 1),
  ("escala", 1),
  ("rapido", 1),
  ("bien", 1),
  ("elegante", 1)
)

In [16]:
spark.stop()


2026-04-22T10:05:35.508289200Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.